# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/data00077/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of one pseudonymized content item for one pseudonymized client on one report date. I will use March 2026 as the analysis window.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 1: Verify the grain

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT
            CONCAT(
                CAST(report_date AS VARCHAR), '|',
                client_hash_id, '|',
                content_hash_id
            )
        ) AS distinct_grain_rows,
        COUNT(*) - COUNT(DISTINCT
            CONCAT(
                CAST(report_date AS VARCHAR), '|',
                client_hash_id, '|',
                content_hash_id
            )
        ) AS duplicate_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""")

print("QUERY 1 — GRAIN")
print(grain_check)

QUERY 1 — GRAIN


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────┬────────────────┐
│ total_rows │ distinct_grain_rows │ duplicate_rows │
│   int64    │        int64        │     int64      │
├────────────┼─────────────────────┼────────────────┤
│    9841378 │             9841378 │              0 │
└────────────┴─────────────────────┴────────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature

- gsc_impressions — historical search visibility available before the decision moment.
- gsc_clicks — historical search traffic available before the decision moment.
- gsc_sum_position — historical average search position available before the decision moment.
- sessions_ai — historical AI-driven sessions available before the decision moment.
- scroll_events — historical engagement available before the decision moment.

Label / proxy

- Future search clicks — the later search-click outcome that the ranking decision is intended to predict.

Context

- report_date — identifies the observation date.
- client_hash_id — identifies the pseudonymized client.
- content_hash_id — identifies the pseudonymized content item.
- gsc_data_available — indicates whether GSC data is available.
- ga4_data_available — indicates whether GA4 data is available.
- month — identifies the warehouse partition.

Excluded

- Future outcome information — excluded because it would not be known at the decision moment and could cause label leakage.
- Client/content IDs — used only for grouping, joining, or splitting, not as model features.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 2: Verify row count and date window

window_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""")

print("\nQUERY 2 — ROW COUNT AND DATE WINDOW")
print(window_check)


QUERY 2 — ROW COUNT AND DATE WINDOW
┌───────────┬─────────────────┬─────────────────┐
│ row_count │ min_report_date │ max_report_date │
│   int64   │      date       │      date       │
├───────────┼─────────────────┼─────────────────┤
│   9841378 │ 2026-03-01      │ 2026-03-31      │
└───────────┴─────────────────┴─────────────────┘



## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 3: Verify data availability using IS TRUE
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
""")

availability_check

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data availability is not complete across the March 2026 slice. Of 9,841,378 rows, 3,611,061 have GSC data available and 413,966 have GA4 data available. This means search or analytics-based analysis may not represent every content item equally, and results should be treated as directional rather than complete coverage

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.